In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ─── 1. LOAD DATA ───────────────────────────────────────────────
df = pd.read_excel("Alumnidata2025.xlsx", header=1)  # row 1 is actual header

print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nFirst 3 rows:\n", df.head(3))

Shape: (260, 20)

Columns:
 ['S. No.', 'Roll No.', 'Student Name', 'Course', 'Discipline', 'Alternative Email', 'Mobile Number', 'Status', 'Company', 'Role', 'Current Posting/Location', 'Institute ', 'Program', 'Joining {Format:Month/Year}', 'Expected Completetion{Format:Month/Year}', 'Remarks', 'Last Update{DD.MM.YYYY}', 'POC', 'Unnamed: 18', 'Unnamed: 19']

First 3 rows:
    S. No.  Roll No.    Student Name Course                        Discipline  \
0       1  12140040  Abhishek Kumar  BTech  Computer Science and Engineering   
1       2  12140060  Aditya Sankhla  BTech  Computer Science and Engineering   
2       3  12140070   Aditya Sharma  BTech  Computer Science and Engineering   

            Alternative Email  Mobile Number           Status Company Role  \
0     kumarabhi2k21@gmail.com     8570072576              NaN     NaN  NaN   
1   adityasankhla14@gmail.com     8369504378  Job - Corporate  Amazon  SDE   
2  adityasharma8224@gmail.com     8224877707              NaN     Na

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ─── 1. LOAD ────────────────────────────────────────────────────
df = pd.read_excel("Alumnidata2025.xlsx", header=1)
print(f"Raw shape: {df.shape}")

# ─── 2. DROP JUNK COLUMNS ───────────────────────────────────────
df.drop(columns=['Unnamed: 18', 'Unnamed: 19'], inplace=True)

# ─── 3. RENAME COLUMNS (clean names) ────────────────────────────
df.columns = [
    'sno', 'roll_no', 'name', 'course', 'discipline',
    'email', 'mobile', 'status', 'company', 'role',
    'location', 'institute', 'program', 'joining_date',
    'completion_date', 'remarks', 'last_update', 'poc'
]

# ─── 4. DROP FULLY EMPTY ROWS ───────────────────────────────────
df.dropna(subset=['name', 'roll_no'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"After dropping empty rows: {df.shape}")

# ─── 5. NORMALIZE TEXT COLUMNS ──────────────────────────────────
for col in ['course', 'discipline', 'status', 'company', 'role', 'location']:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace({'Nan': np.nan, '-': np.nan})

# ─── 6. NORMALIZE COURSE VARIANTS ───────────────────────────────
df['course'] = df['course'].replace({
    'Btech (Hons)': 'BTech (Hons)',
    'Btech':        'BTech',
    'Mtech':        'MTech',
    'Msc':          'MSc',
    'Idd (Btech/Mtech)': 'IDD',
    'Phd':          'PhD',
})

# ─── 7. FILL MISSING STATUS ─────────────────────────────────────
df['status'] = df['status'].fillna('Unknown')

# Standardize status values
status_map = {
    'Job - Corporate':         'Job - Corporate',
    'Job - Government':        'Job - Government',
    'Higher Studies':          'Higher Studies',
    'Misc.':                   'Misc',
    'Misc':                    'Misc',
    'Not Reachable/Responding':'Not Reachable',
    'Unknown':                 'Unknown',
}
df['status'] = df['status'].map(status_map).fillna('Unknown')

# ─── 8. BROAD OUTCOME COLUMN (for modelling later) ──────────────
def broad_outcome(s):
    if s == 'Job - Corporate':   return 'Placed'
    if s == 'Job - Government':  return 'Placed'
    if s == 'Higher Studies':    return 'Higher Studies'
    return 'Other'

df['outcome'] = df['status'].apply(broad_outcome)

# ─── 9. CONVERT EXCEL DATE SERIALS ──────────────────────────────
def fix_date(val):
    try:
        v = float(val)
        if v > 1000:   # looks like an Excel serial
            return pd.Timestamp('1899-12-30') + pd.Timedelta(days=int(v))
        return pd.NaT
    except:
        try:
            return pd.to_datetime(val, dayfirst=True, errors='coerce')
        except:
            return pd.NaT

df['joining_date']    = df['joining_date'].apply(fix_date)
df['completion_date'] = df['completion_date'].apply(fix_date)

# ─── 10. FIX MOBILE (keep as string, drop obviously bad ones) ───
df['mobile'] = df['mobile'].astype(str).str.replace(r'\D', '', regex=True)
df['mobile'] = df['mobile'].apply(lambda x: x if len(x) == 10 else np.nan)

# ─── 11. MISSING VALUE SUMMARY ──────────────────────────────────
print("\n── Missing values per column ──")
print(df.isnull().sum())

# ─── 12. DUPLICATES ─────────────────────────────────────────────
dups = df.duplicated(subset=['roll_no']).sum()
print(f"\nDuplicate roll numbers: {dups}")
df.drop_duplicates(subset=['roll_no'], keep='first', inplace=True)

# ─── 13. BASIC STATS ────────────────────────────────────────────
print("\n── Status distribution ──")
print(df['status'].value_counts())
print("\n── Course distribution ──")
print(df['course'].value_counts())
print("\n── Outcome distribution ──")
print(df['outcome'].value_counts())

# ─── 14. SAVE CLEAN DATA ────────────────────────────────────────
df.to_csv("alumni_cleaned.csv", index=False)
print("\n✅ Saved: alumni_cleaned.csv")
print(f"Final shape: {df.shape}")

Raw shape: (260, 20)
After dropping empty rows: (260, 18)

── Missing values per column ──
sno                  0
roll_no              0
name                 0
course               0
discipline           0
email                0
mobile               0
status               0
company            165
role               166
location           152
institute          189
program            189
joining_date       237
completion_date    236
remarks            166
last_update         87
poc                  0
outcome              0
dtype: int64

Duplicate roll numbers: 0

── Status distribution ──
status
Job - Corporate     90
Unknown             71
Misc                47
Higher Studies      25
Not Reachable       24
Job - Government     3
Name: count, dtype: int64

── Course distribution ──
course
BTech           156
MTech            59
MSc              26
BTech (Hons)     14
PhD               4
IDD               1
Name: count, dtype: int64

── Outcome distribution ──
outcome
Other             